In [1]:
WIDTH = 6
HEIGHT = 6
NUM_AGENTS = 2
NUM_APPLES = 5
GAMMA = 0.99
MC_SAMPLES = 100
MC_DEPTH = 100

In [2]:
import numpy as np
import copy
from tqdm import tqdm
import sys
sys.path.append("../../")  # To allow imports from parent directory


In [3]:
from tadd_helpers.env_functions import State

# --- Import Project Modules ---
from teleport_dynamic.rewards_decentralized import get_reward_minus1_2
from teleport_dynamic.analytical import get_exact_value
from teleport_dynamic.orchard_generator import init_fixed_apples, teleport_agent

In [4]:

# Set Global Seed for reproducibility
SEED = 42
np.random.seed(SEED)

print("Imports successful.")

Imports successful.


In [5]:
def randomize_agents(state):
    """Helper to randomize agent positions for testing arbitrary states."""
    H, W = state.H, state.L
    num_agents = len(state._agents)
    for i in range(num_agents):
        r = np.random.randint(0, H)
        c = np.random.randint(0, W)
        state.set_agent_position(i, np.array([r, c]))

def run_monte_carlo_simulation(
    initial_state: State, 
    initial_actor_idx: int, 
    self_agent_idx: int, 
    reward_func, 
    gamma: float, 
    depth: int, 
    num_samples: int
):
    """
    Runs Monte Carlo rollouts to estimate V(s) for the 'Self' agent.
    """
    total_return = 0.0
    num_agents = len(initial_state._agents)
    
    # Progress bar
    iterator = range(num_samples)
    
    for _ in iterator:
        # Create a working copy for this episode
        # This is necessary because the trajectory modifies agent positions
        episode_state = initial_state.copy()
        current_actor = initial_actor_idx
        
        episode_ret = 0.0
        current_gamma_pow = 1.0 # 1, gamma, gamma^2 ...
        
        for _ in range(depth):
            # 1. Get Reward R(s)
            # Note: reward_func must be pure (no state mod)
            rewards = reward_func(episode_state, current_actor)
            r = rewards[self_agent_idx]
            episode_ret += current_gamma_pow * r
            
            # 2. Transition: Current Actor Teleports
            # Uses the new helper function
            teleport_agent(episode_state, current_actor)
            
            # 3. Next Step: New Actor Selected (Uniform Random)
            current_actor = np.random.randint(0, num_agents)
            
            # 4. Decay
            current_gamma_pow *= gamma
            
            # Optimization: Stop if contribution is negligible
            if current_gamma_pow < 1e-6: 
                break
                
        total_return += episode_ret
        
    return total_return / num_samples


In [6]:


# 1. Setup Environment (Fixed Apples)
base_state = init_fixed_apples(WIDTH, HEIGHT, NUM_AGENTS, NUM_APPLES)

print(f"--- Verification Config ---")
print(f"Grid: {WIDTH}x{HEIGHT}, Agents: {NUM_AGENTS}, Apples: {NUM_APPLES}")
print(f"Reward Func: Decentralized -1/2")
print(f"Gamma: {GAMMA}")
print(f"MC Samples: {MC_SAMPLES}\n")

print(f"{'ID':<4} | {'State Type':<12} | {'Analytical':<10} | {'Monte Carlo':<12} | {'Diff':<8} | {'% Error'}")
print("-" * 80)

errors = []

# Test on 10 Random States
for i in range(3):
    # Generate Random Configuration
    test_state = base_state.copy()
    randomize_agents(test_state)
    actor_idx = np.random.randint(0, NUM_AGENTS)
    
    # We track Agent 0 (Self)
    subject_idx = 0
    
    # 1. Analytical Calculation
    v_exact = get_exact_value(
        test_state, 
        actor_idx, 
        subject_idx, 
        get_reward_minus1_2, 
        GAMMA
    )
    
    # 2. Monte Carlo Estimation
    v_mc = run_monte_carlo_simulation(
        test_state, 
        actor_idx, 
        subject_idx, 
        get_reward_minus1_2, 
        GAMMA, 
        MC_DEPTH, 
        MC_SAMPLES
    )
    
    # Stats
    diff = abs(v_exact - v_mc)
    pct_err = (diff / abs(v_exact)) * 100 if v_exact != 0 else 0.0
    errors.append(pct_err)
    
    # Identify State Type for logging (Is Actor on Apple?)
    actor_pos = test_state.agent_position(actor_idx)
    is_hit = test_state.apples[actor_pos[0], actor_pos[1]] > 0
    state_type = "ACTOR HIT" if is_hit else "MISS"
    
    print(f"{i:<4} | {state_type:<12} | {v_exact:<10.4f} | {v_mc:<12.4f} | {diff:<8.4f} | {pct_err:.2f}%")

--- Verification Config ---
Grid: 6x6, Agents: 2, Apples: 5
Reward Func: Decentralized -1/2
Gamma: 0.99
MC Samples: 100

ID   | State Type   | Analytical | Monte Carlo  | Diff     | % Error
--------------------------------------------------------------------------------
0    | MISS         | 7.0111     | 4.6397       | 2.3714   | 33.82%
1    | ACTOR HIT    | 5.6027     | 3.6438       | 1.9589   | 34.96%
2    | MISS         | 6.6027     | 4.8116       | 1.7911   | 27.13%
